# Pharokka-based Biological Feature Extraction

This notebook must be run on **Google Colab** rather than on a local machine, because **Pharokka and its dependencies can be difficult to install and execute reliably in a local environment**. Google Colab provides a more practical and reproducible setup for running this workflow. Before running the notebook, mount Google Drive in Colab and create the following directories:


- `/content/drive/MyDrive/phage_project`
- `/content/drive/MyDrive/phage_project/genomes`
- `/content/drive/MyDrive/phage_project/pharokka`
- `/content/drive/MyDrive/phage_project/pharokka_db`

In addition, the required input files must be uploaded before executing the notebook. The file **`PA.csv`** must be copied directly into the main `phage_project` folder:

`/content/drive/MyDrive/phage_project/PA.csv`

All **phage genome FASTA files** must be copied into the `genomes` folder:

`/content/drive/MyDrive/phage_project/genomes/`

In summary, `PA.csv` must be placed inside `phage_project`, while all phage FASTA files must be placed inside `phage_project/genomes`. This notebook then runs the Pharokka-based annotation workflow on the uploaded phage genomes and extracts the biological features required for the downstream clustering analysis. If the directories are not created correctly or the input files are not placed in the required locations, the notebook will not run properly.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/phage_project")
INPUT_DIR = BASE / "genomes"
OUTPUT_ROOT = BASE / "pharokka"
DB_DIR = BASE / "pharokka_db"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DB_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DB_DIR:", DB_DIR)
print("FASTA count:", len(list(INPUT_DIR.glob("*.fasta"))))

In [ ]:
%%bash
set -euxo pipefail

# download installer
wget -qnc -O Miniforge3.sh \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh

# install (overwrite any partial install)
rm -rf /usr/local/miniforge
bash Miniforge3.sh -b -p /usr/local/miniforge

# sanity check
/usr/local/miniforge/bin/conda --version

In [ ]:
%%bash
set -e
CONDA="/usr/local/miniforge/bin/conda"

$CONDA config --set channel_priority strict
$CONDA config --remove-key channels || true
$CONDA config --add channels conda-forge
$CONDA config --add channels bioconda

$CONDA config --show channels

In [ ]:
%%bash
set -e
CONDA="/usr/local/miniforge/bin/conda"
$CONDA install -y mamba
/usr/local/miniforge/bin/mamba --version

In [ ]:
%%bash
set -e
CONDA="/usr/local/miniforge/bin/conda"
MAMBA="/usr/local/miniforge/bin/mamba"

$CONDA env remove -n pharokkaENV -y || true
$MAMBA create -y -n pharokkaENV -c conda-forge -c bioconda python=3.10 pharokka prodigal

$CONDA run -n pharokkaENV pharokka.py --version
$CONDA run -n pharokkaENV prodigal -v 2>&1 | head -n 2
echo "✅ pharokkaENV ready"

In [ ]:
import os, shutil
local_db = "/content/pharokka_db"

# clean local target
shutil.rmtree(local_db, ignore_errors=True)

cmd = "/usr/local/miniforge/bin/conda run -n pharokkaENV install_databases.py -o /content/pharokka_db"
print("Running:", cmd)
os.system(cmd)

In [ ]:
import shutil
shutil.copytree("/content/pharokka_db",
                "/content/drive/MyDrive/phage_project/pharokka_db",
                dirs_exist_ok=True)
print("Copied DB to Drive.")

In [ ]:
from pathlib import Path

drive_db = Path("/content/drive/MyDrive/phage_project/pharokka_db")
local_db = Path("/content/pharokka_db")
print("Drive DB exists:", drive_db.exists(), "items:", len(list(drive_db.glob("*"))) if drive_db.exists() else 0)
print("Local DB exists:", local_db.exists(), "items:", len(list(local_db.glob("*"))) if local_db.exists() else 0)

In [ ]:
import subprocess
from pathlib import Path
import tqdm

CONDA = "/usr/local/miniforge/bin/conda"

fasta_paths = sorted(Path(INPUT_DIR).glob("*.fasta"))
print("FASTA files:", len(fasta_paths))

failed = []

for fasta in tqdm.tqdm(fasta_paths, desc="Running Pharokka"):
    sample = fasta.stem
    outdir = Path(OUTPUT_ROOT) / sample
    outdir.mkdir(parents=True, exist_ok=True)

    cmd = [
        CONDA, "run", "-n", "pharokkaENV",
        "pharokka.py",
        "-d", str(DB_DIR),
        "-i", str(fasta),
        "-t", "4",
        "-o", str(outdir),
        "-p", sample,
        "-l", sample,
        "-g", "prodigal",
        "--fast", "-f"
    ]

    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"\n❌ FAILED: {sample}")
        print(r.stderr[-2000:])
        failed.append(sample)

print("Failures:", len(failed))
print(failed)

In [ ]:
missing = []
for fasta in INPUT_DIR.glob("*.fasta"):
    sample = fasta.stem
    hits = list((OUTPUT_ROOT / sample).glob("*_length_gc_cds_density.tsv"))
    if not hits:
        missing.append(sample)

print("Missing *_length_gc_cds_density.tsv:", len(missing))
print(missing)

In [ ]:
import shutil
BASE = Path("/content/drive/MyDrive/phage_project")
zip_path = shutil.make_archive(str(BASE / "pharokka_outputs"), "zip", str(OUTPUT_ROOT))
print("Created:", zip_path)